# 🚨 Lightweight CNN-Based Disaster Detection
## Training Notebook

This notebook walks through the complete training pipeline for the
**Lightweight CNN-Based Disaster Detection Framework** using **MobileNetV2**
transfer learning.

**Classes:** Fire · Flood · Landslide · Earthquake damage · Normal

**Sections:**
1. Environment Setup
2. Dataset Preparation & Preprocessing
3. Data Augmentation & Visualisation
4. Model Architecture (MobileNetV2 + Custom Head)
5. Phase 1 Training — Head Only
6. Phase 2 — Fine-tuning
7. Evaluation — Accuracy, Precision, Recall, F1, Confusion Matrix
8. TFLite Conversion & Benchmark
9. Single-Image Inference Demo

## 1. Environment Setup

In [ ]:
# Install missing libraries if not already present
# !pip install tensorflow scikit-learn matplotlib seaborn streamlit pillow tqdm

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# Add project root to sys.path so we can import from src/
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPUs available     : {tf.config.list_physical_devices("GPU")}')

# Configure GPU memory growth to avoid OOM errors
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR      = PROJECT_ROOT
RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')

os.makedirs(MODELS_DIR, exist_ok=True)
print(f'Project root: {BASE_DIR}')

## 2. Dataset Preparation & Preprocessing

We split the raw images (one folder per class) into **train / val / test** sets
with a **70 / 15 / 15** ratio to ensure unbiased evaluation.

The split is deterministic (fixed random seed) for reproducibility.

In [ ]:
from src.preprocessing import split_dataset, CLASS_NAMES

# ── Split raw data into train / val / test ────────────────────────────────
# This copies images into data/processed/{train,val,test}/<class>/
split_dataset(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
)

print(f'Class names: {CLASS_NAMES}')

In [ ]:
# Count images per split per class and visualise as a bar chart
from pathlib import Path

split_counts = {split: {} for split in ['train', 'val', 'test']}

for split in ['train', 'val', 'test']:
    for cls in CLASS_NAMES:
        cls_dir = Path(PROCESSED_DIR) / split / cls
        n = len(list(cls_dir.glob('*.jpg'))) + len(list(cls_dir.glob('*.png'))) if cls_dir.exists() else 0
        split_counts[split][cls] = n

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(CLASS_NAMES))
width = 0.25
colors = ['#3498db', '#2ecc71', '#e74c3c']

for i, (split, color) in enumerate(zip(['train', 'val', 'test'], colors)):
    counts = [split_counts[split][c] for c in CLASS_NAMES]
    ax.bar(x + i * width, counts, width, label=split.capitalize(), color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES])
ax.set_ylabel('Image Count')
ax.set_title('Dataset Distribution per Split')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Data Augmentation & Visualisation

Augmentation is applied **only to the training set** to artificially increase
dataset diversity and reduce over-fitting.

Augmentations applied:
- **Rotation** ±20°
- **Horizontal flip**
- **Zoom** ±10%
- **Width/Height shift** ±10%

All images are **rescaled to [0, 1]** by dividing pixel values by 255.

In [ ]:
from src.preprocessing import get_data_generators

# Create generators
train_gen, val_gen, test_gen = get_data_generators(PROCESSED_DIR)

print(f'Training samples   : {train_gen.samples}')
print(f'Validation samples : {val_gen.samples}')
print(f'Test samples       : {test_gen.samples}')
print(f'Class indices      : {train_gen.class_indices}')

In [ ]:
# Visualise one batch of augmented training images
images, labels = next(train_gen)
class_idx_to_name = {v: k for k, v in train_gen.class_indices.items()}

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Augmented Training Images (first 32)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(class_idx_to_name[np.argmax(labels[i])], fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Model Architecture

We use **MobileNetV2** as the backbone with its ImageNet weights.
The classification head is replaced with our own dense layers for the 5 disaster classes.

```
Input (224×224×3) → MobileNetV2 backbone (frozen) → 
GlobalAveragePooling → Dense(256) → Dropout(0.4) → 
Dense(128) → Dropout(0.3) → Dense(5, softmax)
```

In [ ]:
from src.model import build_model, compile_model, model_summary

# Build the MobileNetV2-based model
model = build_model()
model = compile_model(model, learning_rate=1e-3)

# Display layer summary and parameter counts
model_summary(model)

In [ ]:
# Visualise the model graph (optional — requires graphviz)
try:
    from tensorflow.keras.utils import plot_model
    plot_model(model, to_file=os.path.join(MODELS_DIR, 'model_architecture.png'),
               show_shapes=True, show_layer_names=True, dpi=96)
    print('Model diagram saved.')
except Exception as e:
    print(f'Could not generate model diagram: {e}')

## 5. Phase 1 Training — Head Only

In Phase 1, only the custom Dense head is trained; the MobileNetV2 backbone
is fully frozen. This converges quickly and establishes a good starting point
before fine-tuning.

Callbacks:
- `ModelCheckpoint` — saves the best val_accuracy model to `models/disaster_model.h5`
- `EarlyStopping` — stops if val_loss doesn't improve for 5 epochs
- `ReduceLROnPlateau` — halves LR after 3 stagnant epochs

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

PHASE1_EPOCHS = 10
checkpoint_path = os.path.join(MODELS_DIR, 'disaster_model.h5')

callbacks_phase1 = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
]

print(f'Training for up to {PHASE1_EPOCHS} epochs (Phase 1 — head only)…')
history1 = model.fit(
    train_gen,
    epochs=PHASE1_EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1,
)

In [ ]:
# Plot Phase 1 training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history1.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
axes[0].set_title('Phase 1 — Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history1.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history1.history['val_loss'], label='Val',   linewidth=2, linestyle='--')
axes[1].set_title('Phase 1 — Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'phase1_history.png'), dpi=150)
plt.show()

## 6. Phase 2 — Fine-tuning

Unfreeze the **top 30 backbone layers** and continue training with a very
small learning rate (1e-5) to adapt domain-specific features without
overwriting ImageNet knowledge.

In [ ]:
from src.model import unfreeze_top_layers

PHASE2_EPOCHS = 10

model = unfreeze_top_layers(model, num_layers_to_unfreeze=30, fine_tune_lr=1e-5)

print(f'Fine-tuning for up to {PHASE2_EPOCHS} epochs…')
history2 = model.fit(
    train_gen,
    epochs=PHASE2_EPOCHS,
    validation_data=val_gen,
    callbacks=callbacks_phase1,  # Reuse same callbacks
    verbose=1,
)

# Save the fine-tuned model
final_path = os.path.join(MODELS_DIR, 'disaster_model_final.h5')
model.save(final_path)
print(f'Final model saved → {final_path}')

In [ ]:
# Plot Phase 2 training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history2.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history2.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
axes[0].set_title('Phase 2 Fine-tuning — Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history2.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history2.history['val_loss'], label='Val',   linewidth=2, linestyle='--')
axes[1].set_title('Phase 2 Fine-tuning — Loss', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'phase2_history.png'), dpi=150)
plt.show()

## 7. Evaluation

Compute **Accuracy, Precision, Recall, F1-Score** on the held-out **test set**.

Plot both a raw-count and a normalised confusion matrix.

In [ ]:
from src.evaluate import evaluate_model

# Load the best checkpoint for evaluation
best_model = tf.keras.models.load_model(checkpoint_path)

metrics = evaluate_model(best_model, test_gen, CLASS_NAMES, save_dir=MODELS_DIR)

print(f"\nFinal Test Accuracy : {metrics['accuracy']:.4f}")
print(f"Weighted F1-Score   : {metrics['f1']:.4f}")

## 8. TFLite Conversion & Benchmark

Convert the trained model to **TensorFlow Lite** with **dynamic-range quantisation**.
This reduces the model size ~4× with minimal accuracy loss.

In [ ]:
from src.convert_tflite import convert_dynamic_range, benchmark_tflite

tflite_path = os.path.join(MODELS_DIR, 'disaster_model.tflite')

# Convert .h5 → .tflite with dynamic quantisation
size_bytes = convert_dynamic_range(checkpoint_path, tflite_path)
print(f'TFLite model: {size_bytes / 1024:.1f} KB')

# Compare original size
h5_size = os.path.getsize(checkpoint_path) / 1024
reduction = (1 - (size_bytes / 1024) / h5_size) * 100
print(f'Original .h5 size : {h5_size:.1f} KB')
print(f'Size reduction    : {reduction:.1f}%')

In [ ]:
# Benchmark the TFLite model's inference speed
bench = benchmark_tflite(tflite_path, num_runs=50)

print(f"\nMean inference time : {bench['mean_ms']:.2f} ms")
print(f"Min                 : {bench['min_ms']:.2f} ms")
print(f"Max                 : {bench['max_ms']:.2f} ms")
print(f"Model size          : {bench['memory_kb']:.1f} KB")

## 9. Single-Image Inference Demo

Run a quick inference on a single image to confirm the pipeline works end-to-end.

Expected output format:
```
Fire detected – Confidence 93%
```

In [ ]:
from src.preprocessing import preprocess_single_image
from pathlib import Path

# ── Replace this path with a real image for testing ────────────────────────
# test_image_path = 'path/to/your/test_image.jpg'

# Auto-pick the first available image from test set for demo
test_image_path = None
for cls in CLASS_NAMES:
    cls_dir = Path(PROCESSED_DIR) / 'test' / cls
    imgs = list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png'))
    if imgs:
        test_image_path = str(imgs[0])
        real_label = cls
        break

if test_image_path is None:
    print('No test images found. Add images to data/raw/ and run split_dataset().')
else:
    # Preprocess & run inference
    img_array = preprocess_single_image(test_image_path)
    probs = best_model.predict(img_array, verbose=0)[0]
    pred_idx   = int(np.argmax(probs))
    pred_class = CLASS_NAMES[pred_idx]
    confidence = float(probs[pred_idx]) * 100

    # Display image and prediction
    from PIL import Image as PILImage
    img = PILImage.open(test_image_path)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].imshow(img)
    axes[0].set_title(f'True label: {real_label}', fontsize=12)
    axes[0].axis('off')

    colors = ['#3498db'] * 5
    colors[pred_idx] = '#e74c3c'
    axes[1].barh(CLASS_NAMES, probs * 100, color=colors)
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Class Probabilities', fontsize=12)
    axes[1].set_xlim(0, 100)
    for i, v in enumerate(probs * 100):
        axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center')

    plt.tight_layout()
    plt.show()

    # ── Output in requested format ────────────────────────────────────────
    result_str = (
        f"{'No disaster' if pred_class == 'normal' else pred_class.replace('_', ' ').title()} "
        f"detected – Confidence {confidence:.0f}%"
    )
    print('\n' + '='*50)
    print(f'  🔍 {result_str}')
    print('='*50)